In [1]:
import argparse
import os
import scanpy as sc
import scvi
import seaborn as sns
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score
import anndata as ad
import cellrank as cr

/opt/conda/envs/py_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 03 - Analysis

## 03.1 - Def

In [2]:
def a03_fun_01_plots1(adata, subsetted=False):
    a = 'full'
    if subsetted: 
        a = 'subset'

    # Plain UMAPs
    fig = sc.pl.umap(
        adata, 
        color=argslevel, 
        title=f'{argsname} - {a} - {argslevel}', 
        show=False, 
        return_fig=True
    )
    fig.savefig(f'{output_dir}/a03_{argslevel}_{argsname}_01.1_{a}_umap.png', 
                bbox_inches="tight")

    # Genes of interest
    fig = sc.pl.umap(
        adata, 
        color=argsgoi, 
        use_raw=True, 
        color_map='Reds', 
        vmax='p99', 
        title=f'{argsname} - {a} - {argsgoi}', 
        show=False, 
        return_fig=True
    )
    fig.savefig(f'{output_dir}/a03_{argslevel}_{argsname}_01.2_{a}_{argsgoi}_umap.png', 
                bbox_inches="tight")

    fig = sc.pl.violin(
        adata, 
        keys=argsgoi, 
        groupby=argslevel, 
        use_raw=True, 
        rotation=90, 
        show=False
    )
    plt.savefig(f'{output_dir}/a03_{argslevel}_{argsname}_01.3_{a}_{argsgoi}_violin.png', 
                bbox_inches="tight")

    # Housekeeping genes
    if isinstance(argshousekeeping_genes, str): argshk = argshousekeeping_genes.split(',')
    valid_hk = [g for g in argshk if g in adata.raw.var_names]
    fig = sc.pl.stacked_violin(
        adata, 
        var_names=valid_hk, 
        groupby='leiden', 
        title=f"{argsname} - {a} - Housekeeping Gene Distribution across Clusters",
        use_raw=True,
        show=False
    )   
    plt.savefig(f'{output_dir}/a03_{argslevel}_{argsname}_01.4_{a}_hk_distribution.png', 
                bbox_inches="tight")
    

In [3]:
def a03_fun_02_relative_expression(adata, subsetted=False):
    a = 'full'
    if subsetted: 
        a = 'subset'
        
    print('--> Relative expression of genes of interest compared to housekeeping baseline')
    if isinstance(argshousekeeping_genes, str): argshk = argshousekeeping_genes.split(',')
    valid_hk = [g for g in argshk if g in adata.raw.var_names]
    
    if valid_hk:
        print(f'--> Calculating background signature score using: {valid_hk}')
        sc.tl.score_genes(adata, gene_list=valid_hk, score_name='hk_signature_score', use_raw=True)
        
        # Gene of interest normalized against the housekeeping signature
        if argsgoi in adata.raw.var_names:
            # Extract the raw expression array for the gene of interest
            # (.X might be sparse, so we convert to a dense 1D array)
            import scipy.sparse as sp
            gene_expr = adata.raw[:, argsgoi].X
            if sp.issparse(gene_expr):
                gene_expr = gene_expr.toarray().flatten()
            else:
                gene_expr = gene_expr.flatten()
            
            # Using Subtraction because adata.raw is typically log1p transformed
            # log(Gene) - log(HK_baseline) = log(Gene / HK_baseline)
            new_col_name = f'{argsgoi}_relative_to_hk_baseline'
            adata.obs[new_col_name] = gene_expr - adata.obs['hk_signature_score']
    
            #print(adata.obs)
    
            fig = sc.pl.umap(
                adata, 
                color=argslevel, 
                title=f'{argsname} - {argslevel}',  
                show=False, 
                return_fig=True
            )
            fig.savefig(f'{output_dir}/a03_{argslevel}_{argsname}_02.1_{a}_umap.png', 
                        bbox_inches="tight")
            
            sc.pl.umap(
                adata, 
                color=new_col_name, color_map='coolwarm', # coolwarm is great for showing over/under expression (diverging) 
                title=f'{argsname} - {a} - {argsgoi} relative to HK baseline',   
                show=False,
                return_fig=True
            )
            fig.savefig(f'{output_dir}/a03_{argslevel}_{argsname}_02.2_{a}_{argsgoi}_relative_to_hk_umap.png', 
                        bbox_inches="tight")
    
            ax = sc.pl.violin(adata, 
                       keys=new_col_name,
                       groupby=argslevel,
                       rotation=90, 
                       show=False)
            ax.set_title(f'{argsname} - {a} - {argsgoi} relative to HK baseline')
            plt.savefig(f"{output_dir}/a03_{argslevel}_{argsname}_02.3_{a}_{argsgoi}_relative_to_hk_violin.png", 
                        bbox_inches="tight")
            print(ax)

In [4]:
def a03_fun_04_DE_genes(adata, subsetted=False):
    a = 'full'
    if subsetted: 
        a = 'subset'
        
    # DE genes
    # 1. Configurazione
    # Assicurati che adata sia già caricato
    # adata = sc.read_h5ad("path/to/your/data.h5ad")
    
    #gene = 'CFH'#'SRCIN1'
    #levels = ['paper_annotation', 'annot_level_1']#, 'annot_level_2', 'annot_level_3_rev2', 'annot_level_4_rev2', 'cell_type']
    
    # Controllo pre-processing (normalizzazione e logaritmo sono necessari per i test statistici)
    if adata.X.max() > 20: 
        print("Il dataset sembra contenere raw counts. Procedo con normalizzazione e log1p...")
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
    
    # Dizionario per salvare i risultati filtrati se vuoi usarli dopo nel notebook
    filtered_results = {}
    
    
    print(f"\n{'='*20} Analisi per: {argslevel} {'='*20}")
    
    # Check se la colonna esiste in adata.obs
    if argslevel not in adata.obs.columns:
        print(f"ATTENZIONE: Colonna '{argslevel}' non trovata in adata.obs. Salto.")
        #continue

    # Filtra i gruppi con meno di N cellule (es. 10)
    counts = adata.obs[argslevel].value_counts()
    valid_groups = counts[counts >= 10].index
    
    # Crea una maschera per usare solo le cellule dei gruppi validi
    mask = adata.obs[argslevel].isin(valid_groups)
    
    # Esegui rank_genes_groups su una vista filtrata
    # Nota: usiamo mask per selezionare le cellule, ma passiamo l'intero adata con use_raw=False se normalizzato
    temp_adata = adata[mask].copy()
    sc.tl.rank_genes_groups(temp_adata, groupby=argslevel, method='wilcoxon', use_raw=True)
    
    # Estrai i risultati da temp_adata invece che da adata
    markers_df = sc.get.rank_genes_groups_df(temp_adata, group=None)

    # 2. Calcolo Geni Differenzialmente Espressi (DE) con FILTRO
    
    # A. Identifica i gruppi con almeno 5 cellule (il minimo per fare statistica)
    counts = adata.obs[argslevel].value_counts()
    valid_groups = counts[counts >= 5].index.tolist()
    
    if len(valid_groups) < 2:
        print(f"Saltato {argslevel}: meno di 2 gruppi validi per il confronto.")
        #continue

    # B. Crea un oggetto temporaneo solo con i gruppi validi
    temp_adata = adata[adata.obs[argslevel].isin(valid_groups)].copy()
    
    # C. Esegui il test SU TEMP_ADATA (non su adata)
    # Nota: usa use_raw=False se hai normalizzato temp_adata o se adata.raw non è settato
    sc.tl.rank_genes_groups(temp_adata, groupby=argslevel, method='wilcoxon', use_raw=True)
    
    # D. Estrai tabella completa DA TEMP_ADATA
    markers_df = sc.get.rank_genes_groups_df(temp_adata, group=None)
    
    # Salva tabella completa
    filename_full = f"markers/markers_{argsname}_full_{argslevel}.csv"
    # markers_df.to_csv(filename_full, index=False)
    # print(f" -> Salvati tutti i marker in: {filename_full}")

    # 3. Filtro per gene di interesse (SRCIN1)
    if argsgoi in adata.raw.var_names:
        
        # A. Calcola l'espressione media del gene per ogni cluster in questo livello
        # Creiamo un piccolo dataframe temporaneo con l'espressione del gene e l'annotazione
        gene_expr = pd.DataFrame({
            'expression': adata.raw.obs_vector(argsgoi), # <--- Prende i dati da raw in modo sicuro
            'cluster': adata.obs[argslevel]
        })
        
        # Media per cluster
        mean_expr = gene_expr.groupby('cluster')['expression'].mean()
        
        # B. Identifica i cluster che esprimono il gene (es. espressione media > 0)
        # Puoi alzare la soglia (es. > 0.1) se vuoi essere più restrittivo
        clusters_expressing_gene = mean_expr[mean_expr > 0].index.tolist()
        
        print(f" -> Cluster che esprimono {argsgoi}: {len(clusters_expressing_gene)} su {len(mean_expr)}")
        # C. Filtra la tabella dei marker
        markers_filtered = markers_df[markers_df['group'].isin(clusters_expressing_gene)]
        markers_filtered = markers_filtered[(markers_filtered['names'] == argsgoi) & (markers_filtered['pvals_adj'] < 5e-2)]
        
        # Opzionale: Filtra per significatività (es. pval_adj < 0.05 e logfoldchange > 1)
        # markers_filtered = markers_filtered[(markers_filtered['pvals_adj'] < 0.05) & (markers_filtered['logfoldchanges'] > 1)]
        
        # Salva tabella filtrata
        filename_filtered = f"{output_dir}/a03_{argslevel}_{argsname}_04_{a}_markers_filtered_for_{argsgoi}.csv"
        markers_filtered.to_csv(filename_filtered, index=False)
        print(f" -> Salvati marker dei cluster rilevanti in: {filename_filtered}")
        
        # Salva nel dizionario per uso immediato nel notebook
        filtered_results[argslevel] = markers_filtered
        
        # Mostra anteprima dei top marker per il primo cluster trovato che esprime il gene
        if not markers_filtered.empty:
            print(f"    Cell types at {argsname} ({a}) expressing {argsgoi} with p-value<0.05 - grouping by {argslevel}':")
            print(markers_filtered)

    else:
        print(f"ERRORE: Il gene {argsgoi} non è presente in adata.raw.var_names.")
            
    print("\nFatto!")

## 03.2 - Run

In [5]:
argsname = '1month'
argslevel = 'annot_level_2'
argsfile = f'/python/Results_ann/expr_{argsname}_annotated.h5ad'
argsgoi = 'SRCIN1'
argshousekeeping_genes = 'RPL13A,RPLP0,ACTB,GAPDH'
argsoutput = 'Results_03-04'

In [6]:
output_dir = argsoutput + '/' + argslevel + '/' + argsname if argsname else argsoutput
os.makedirs(output_dir, exist_ok=True)
sc.settings.figdir = output_dir
sc.settings._vector_friendly = False

print(f'--> Loading data from {argsfile}')
adata = sc.read_h5ad(argsfile)
print(f'--> Data loaded from {argsfile}')

--> Loading data from /python/Results_ann/expr_1month_annotated.h5ad
--> Data loaded from /python/Results_ann/expr_1month_annotated.h5ad


In [ ]:
a03_fun_01_plots1(adata, subsetted=False)
a03_fun_02_relative_expression(adata, subsetted=False)

--> Relative expression of genes of interest compared to housekeeping baseline
--> Calculating background signature score using: ['RPL13A', 'RPLP0', 'ACTB', 'GAPDH']


In [ ]:
print(f'--> Subsetting adata for {argsname}')
cells_to_remove = ['EC', 'MC', 'NC Derivatives', 'CP', 'PSC', 'Microglia']
mask = (~adata.obs[argslevel].isin(cells_to_remove)) & (~adata.obs['leiden'].isin(cells_to_remove))
adata = adata[mask]

a03_fun_01_plots1(adata, subsetted=True)
a03_fun_02_relative_expression(adata, subsetted=True)

In [ ]:
a03_fun_04_DE_genes(adata)

In [ ]:
for argsname in ['1month', '2month', '3month', '4month', '5month', '6month']:
    output_dir = argsoutput + '/' + argslevel + '/' + argsname if argsname else argsoutput
    os.makedirs(output_dir, exist_ok=True)
    sc.settings.figdir = output_dir
    sc.settings._vector_friendly = False

    print(f'--> Loading data from {argsfile}')
    adata = sc.read_h5ad(argsfile)
    print(f'--> Data loaded from {argsfile}')
    
    a03_fun_01_plots1(adata, subsetted=False)
    a03_fun_02_relative_expression(adata, subsetted=False)

    print(f'--> Subsetting adata for {argsname}')
    cells_to_remove = ['EC', 'MC', 'NC Derivatives', 'CP', 'PSC', 'Microglia']
    mask = (~adata.obs[argslevel].isin(cells_to_remove)) & (~adata.obs['leiden'].isin(cells_to_remove))
    adata = adata[mask]
    
    a03_fun_01_plots1(adata, subsetted=True)
    a03_fun_02_relative_expression(adata, subsetted=True)

    # De genes

    a03_fun_04_DE_genes(adata)
    

# 04 - Trajectory inference

## 04.1 - Def

In [ ]:
def ti_04_fun_01(adata):
    # 4. Identify Highly Variable Genes across batches
    target_genes = min(2000, adata.n_vars - 1)
    # DO NOT set subset=True yet. We need to save the raw matrix first.
    sc.pp.highly_variable_genes(
        adata, 
        flavor="seurat_v3", 
        batch_key="timepoint", 
        n_top_genes=target_genes, 
        subset=False 
    )
    adata.raw = adata
    # Now subset to the 2000 HVGs for scVI
    adata = adata[:, adata.var.highly_variable].copy()
    
    print(f'--> Number of highly variable genes selected: {adata.n_vars}')

    return adata

In [ ]:
def ti_04_fun_02_scvi(adata, argsname, argslevel):
    # scVI integration
    print(f'--> Running scVI integration')
    # 1. Setup AnnData for scVI
    print(f'--> 01 - Setting up AnnData for scVI')
    scvi.model.SCVI.setup_anndata(adata, batch_key="timepoint")
    
    print(f'--> 02 - Training scVI model')
    
    # 2. Initialize and Train Model
    import warnings
    # Suppress the specific distribution warning
    warnings.filterwarnings("ignore", message=".*support of the distribution.*")
    model = scvi.model.SCVI(adata, n_latent=30, n_layers=2)
    model.train(early_stopping=True)
    # Optional: Restore normal warning behavior after training
    warnings.filterwarnings("default", message=".*support of the distribution.*")
    
    # 3. Extract the corrected representation
    print(f'--> 03 - Extracting scVI latent representation and recomputing UMAP')
    adata.obsm["X_scVI"] = model.get_latent_representation()
    # Recompute the manifold
    # Use the scVI representation for the neighborhood graph
    sc.pp.neighbors(adata, use_rep="X_scVI")
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=1.0)
    print(f'--> UMAP embedding computed with {adata.obsm["X_umap"].shape[0]} cells')
    # Visualize to check if integration worked
    print(f'--> 03 - Visualizing UMAP embedding')
    fig = sc.pl.umap(
        adata, 
        color=["timepoint", "leiden", argslevel], 
        wspace=0.4,
        show=False
    )
    fig = plt.gcf()
    fig.savefig(f'{output_dir}/a04_{argslevel}_{argsname}_02_organoids_integrated_umap.png', 
                bbox_inches="tight")

    return adata

In [ ]:
def ti_04_fun_03_TI(adata, argsname, argslevel):
    # 1. Run PAGA on your scVI-integrated graph to get cluster connectivity
    sc.tl.paga(adata, groups=argslevel)
    # 2. Initialize the CytoTRACE kernel (calculates developmental direction)
    # It automatically uses the neighbors computed from your scVI integration
    ctk = cr.kernels.CytoTRACEKernel(adata).compute_cytotrace(layer="X")
    ctk.compute_transition_matrix()
    # 3. Initialize the estimator to find starting and ending points
    estimator = cr.estimators.GPCCA(ctk)
    
    # Find and plot all macrostates
    estimator.compute_macrostates(n_states=6, cluster_key=argslevel)
    estimator.plot_macrostates(
        which="all", 
        show=False
    )
    fig = plt.gcf()
    fig.savefig(f'{output_dir}/a04__{argslevel}_{argsname}_03.1_macrostates.png',
                bbox_inches="tight")
    
    estimator.predict_terminal_states()
    # Plot the predicted terminal states
    estimator.plot_macrostates(
        which="terminal",
        show=False
    )
    fig = plt.gcf()
    fig.savefig(f'{output_dir}/a04_{argslevel}_{argsname}_03.2_predicted_terminal_states.png', 
                bbox_inches="tight")
    
    # Compute fate probabilities bypassing OpenMPI to avoid memory/pipe crashes
    estimator.compute_fate_probabilities(
        solver="gmres", 
        use_petsc=False, 
        tol=1e-6,
        preconditioner='ilu'
    )

    return adata

In [ ]:
def ti_04_fun_04_SRCIN1(adata, argsname, argslevel):
    # 1. Restore the full dataset (cells x 22962 genes) from .raw
    adata_full = adata.raw.to_adata()
    
    # 2. Safely carry over the CellRank trajectory metadata and transition matrices
    adata_full.uns = adata.uns.copy()
    adata_full.obsp = adata.obsp.copy()
    adata_full.obsm = adata.obsm.copy() # Add this to retain scVI/CellRank embeddings
    adata_full.obs = adata.obs.copy()
    
    # 3. Overwrite the active adata object
    adata = adata_full
    
    # 4. Normalize and log-transform the full counts
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    cr.pl.gene_trends(
        adata,
        model=cr.models.GAM(adata),
        genes=['SRCIN1'], 
        time_key="ct_pseudotime", # Specifies the x-axis
        lineages=["Dorsal Telencephalic Neuron"], # Optional: remove to plot all lineages
        weight_threshold=0.2, 
        show_progress_bar=True
    )
    fig = plt.gcf()
    fig.savefig(f'{output_dir}/a04_{argslevel}_{argsname}_04_SRCIN1_gene_trend.png', 
                bbox_inches="tight")
    
    return adata

## 04.2 - Per month

In [ ]:
save_dir = 'Adata_save'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
location='/python/Results_ann'
argslevel= 'annot_level_2'
for argsname in ['1month', '2month', '3month', '4month', '5month', '6month']:
    output_dir = argsoutput + '/' + argslevel + '/' + argsname if argsname else argsoutput
    os.makedirs(output_dir, exist_ok=True)
    sc.settings.figdir = output_dir
    sc.settings._vector_friendly = False
    
    adata = sc.read_h5ad(f"{location}/expr_{argsname}_annotated.h5ad")
    if adata.raw is not None:
        adata = adata.raw.to_adata()
    adata.obs["timepoint"] = "1month"

    # Functions
    # 01
    print('--> ti - fun 01')
    adata = ti_04_fun_01(adata=adata)
    
    # 02
    print('--> ti - fun 02')
    adata = ti_04_fun_02_scvi(adata=adata, argsname=argsname, argslevel=argslevel)
    
    # Save tmp
    print('--> ti - save tmp')
    adata.write_h5ad(f'{save_dir}/organoids_trajectory_inference_{argsname}_tmp.h5ad')

    # Directed Trajectory Inference
    print('--> ti - fun 03')
    adata = ti_04_fun_03_TI(adata=adata, argsname=argsname, argslevel=argslevel)

    print('--> ti - fun 04')
    # adata = ti_04_fun_04_SRCIN1(adata=adata, argsname=argsname, argslevel=argslevel)
    
    # Save Final
    print('--> ti - save final')
    # adata.write_h5ad(f'{save_dir}/organoids_trajectory_inference_{argsname}_final.h5ad')

## 04.3 - 1-3

In [ ]:
save_dir = 'Adata_save'
os.makedirs(output_dir, exist_ok=True)
location = '/python/Results_ann'
argslevel = 'annot_level_2'
argsname = '1-3month'

output_dir = argsoutput + '/' + argslevel + '/' + argsname if argsname else argsoutput
os.makedirs(output_dir, exist_ok=True)
sc.settings.figdir = output_dir
sc.settings._vector_friendly = False

# 1. Load your preprocessed datasets
adata_m1 = sc.read_h5ad(f"{location}/expr_1month_annotated.h5ad")
if adata_m1.raw is not None:
    adata_m1 = adata_m1.raw.to_adata()
adata_m2 = sc.read_h5ad(f"{location}/expr_2month_annotated.h5ad")
if adata_m2.raw is not None:
    adata_m2 = adata_m2.raw.to_adata()
adata_m3 = sc.read_h5ad(f"{location}/expr_3month_annotated.h5ad")
if adata_m3.raw is not None:
    adata_m3 = adata_m3.raw.to_adata()


adata_m1.obs["timepoint"] = "1month"
adata_m2.obs["timepoint"] = "2month"
adata_m3.obs["timepoint"] = "3month"

# 3. Concatenate
adata = ad.concat(
    [adata_m1, adata_m2, adata_m3], 
    label="batch", 
    keys=["1month", "2month", "3month"], 
    join="inner" # keeps only intersecting genes
)
print(f'--> Concatenated dataset shape: {adata.shape}')
print(f'--> Number of genes after intersection: {adata.n_vars}')


# Functions
# 01
print('--> ti - fun 01')
adata = ti_04_fun_01(
    adata=adata
)

# 02
print('--> ti - fun 02')
adata = ti_04_fun_02_scvi(
    adata=adata, 
    argsname=argsname, 
    argslevel=argslevel
)

# Save tmp
print('--> ti - save tmp')
adata.write_h5ad(f'{save_dir}/organoids_trajectory_inference_{argsname}_tmp.h5ad')

# Directed Trajectory Inference
print('--> ti - fun 03')
adata = ti_04_fun_03_TI(
    adata=adata, 
    argsname=argsname, 
    argslevel=argslevel
)

print('--> ti - fun 04')
# adata = ti_04_fun_04_SRCIN1(adata=adata, argsname=argsname, argslevel=argslevel)

# Save Final
print('--> ti - fun final')
# adata.write_h5ad(f'{save_dir}/organoids_trajectory_inference_{argsname}_final.h5ad')